# Quality pilot session evidence
Snapshot of actual MCP cells and outputs on 15 October. This includes diagnostic failures and asynchronous process launches; do not Run All. Read quality_experiments.md for reproducible commands.


In [2]:
from pathlib import Path
import subprocess, os, json
repo = Path('/content/hse-nlp')
if not repo.exists():
    subprocess.run(['git','clone','--depth=1','--branch=2026','https://github.com/OsVox/hse-nlp.git',str(repo)],check=True)
os.chdir(repo)
print(subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
subprocess.run(['pip','-q','install','transformers>=4.53,<5','pyarrow>=18','requests>=2.31'],check=True)
print('Setup complete')

29e07841685c7a800f3dce8c5d4c800fa80f26bb
Setup complete


In [3]:
from pathlib import Path
import json, hashlib, random
import pyarrow.parquet as pq
from transformers import PreTrainedTokenizerFast
from homeworks.hw2.data import training_shard, validation_examples, fetch_parquet
cache = Path('/content/hw2_cache')
tok = PreTrainedTokenizerFast.from_pretrained('homeworks/hw2/model')
rows = validation_examples(cache)
test = pq.read_table(fetch_parquet('data/test/test.parquet',cache),columns=['question']).to_pylist()
summary = {'gpu': torch.cuda.get_device_name(0), 'model_sha256':hashlib.sha256(Path('homeworks/hw2/model/model.safetensors').read_bytes()).hexdigest()}
for name, data in [('heldout',rows),('test_questions',test)]:
    lengths = [len(x) for x in tok([r['question'] for r in data])['input_ids']]
    summary[name] = {'rows':len(data),'source_over_128':sum(x>128 for x in lengths),'source_max':max(lengths)}
codes = [r['code'] for r in rows]
ids = tok(codes)['input_ids']
summary['heldout'].update(target_over_192=sum(len(x)>192 for x in ids),target_max=max(map(len,ids)),roundtrip_exact=sum(a==b for a,b in zip(codes,tok.batch_decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=False))))
print(json.dumps(summary,indent=2))
for i in [0,255,512,1023]:
 print('HELDOUT',i,repr(rows[i]['question']),repr(rows[i]['code']))
print('RUNTIME_FILES', [str(p) for p in Path('/content').iterdir()])

{
  "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "model_sha256": "987a9f2b0837a718430704887d1c05bda05c0479d86d8570df14d8275b7fbaa0",
  "heldout": {
    "rows": 1024,
    "source_over_128": 2,
    "source_max": 159,
    "target_over_192": 5,
    "target_max": 325,
    "roundtrip_exact": 1024
  },
  "test_questions": {
    "rows": 1919,
    "source_over_128": 3,
    "source_max": 166
  }
}
HELDOUT 0 'Mark has 10 crayons. He gives 2 crayons to his younger sister and loses another 4 while he was playing. How many crayons does Mark have left?' 'def solve() -> int:\n    crayonsTotal = 10\n    crayonsGiven = 2\n    crayonsLost = 4\n    crayonsLeft = crayonsTotal - crayonsGiven - crayonsLost\n    result = crayonsLeft\n\n    return result\n'
HELDOUT 255 'Lily and Sam were painting pictures for a school project. They had two cans of paint, one red and one blue. Each can had 500 ml of paint. Lily used 150 ml of red paint and 100 ml of blue paint. Sam mixed 100 ml of red paint and 200 

In [4]:
import ast, collections
stats = []
for shard in [0,15,30,45,55]:
    table = pq.read_table(training_shard(shard,cache), columns=['question','code'])
    sample = table.slice(2000,1000).to_pylist()
    source_lens = list(map(len,tok([r['question'] for r in sample])['input_ids']))
    target_lens = list(map(len,tok([r['code'] for r in sample])['input_ids']))
    stats.append({'shard':shard,'rows':len(table),'source_over_128':sum(n>128 for n in source_lens),'target_over_192':sum(n>192 for n in target_lens),'mean_source':sum(source_lens)/1000,'mean_target':sum(target_lens)/1000})
print(json.dumps(stats,indent=2))
print('SAMPLE',repr(sample[0]))

[
  {
    "shard": 0,
    "rows": 200000,
    "source_over_128": 2,
    "target_over_192": 8,
    "mean_source": 43.977,
    "mean_target": 65.964
  },
  {
    "shard": 15,
    "rows": 200000,
    "source_over_128": 0,
    "target_over_192": 2,
    "mean_source": 40.182,
    "mean_target": 62.557
  },
  {
    "shard": 30,
    "rows": 200000,
    "source_over_128": 0,
    "target_over_192": 48,
    "mean_source": 54.227,
    "mean_target": 93.17
  },
  {
    "shard": 45,
    "rows": 200000,
    "source_over_128": 0,
    "target_over_192": 3,
    "mean_source": 39.967,
    "mean_target": 61.842
  },
  {
    "shard": 55,
    "rows": 200000,
    "source_over_128": 8,
    "target_over_192": 91,
    "mean_source": 70.706,
    "mean_target": 118.948
  }
]
SAMPLE {'question': "Emma wants to ride the Ferris wheel 2 times, the Tilt-A-Whirl 3 times, and the teacups once. It costs 2 tickets to ride the Ferris wheel, 4 tickets to ride the Tilt-A-Whirl, and 3 tickets to ride the teacups. Emma's favo

In [5]:
table0 = pq.read_table(training_shard(0,cache),columns=['question'])['question'].to_pylist()
heldout_questions = {r['question'] for r in rows}
print('heldout_question_duplicates_in_remaining_shard0', sum(q in heldout_questions for q in table0[1024:]))
print('tokenizer_cleanup',tok.clean_up_tokenization_spaces)
print('current_gpu_memory', torch.cuda.mem_get_info())

heldout_question_duplicates_in_remaining_shard0 24
tokenizer_cleanup False
current_gpu_memory (101388779520, 101974081536)


In [6]:
import subprocess
subprocess.run(['git','pull','--ff-only'],check=True)
subprocess.run(['python','-m','pytest','-q','homeworks/hw2/tests/test_data.py','homeworks/hw2/tests/test_speed.py'],check=True)
subprocess.run(['python','-m','homeworks.hw2.numeric_eval','--model-dir','homeworks/hw2/model','--output','/content/quality/baseline_numeric.json'],check=True)

CalledProcessError: Command '['python', '-m', 'pytest', '-q', 'homeworks/hw2/tests/test_data.py', 'homeworks/hw2/tests/test_speed.py']' returned non-zero exit status 1.

In [7]:
result = subprocess.run(['python','-m','pytest','-q','homeworks/hw2/tests/test_data.py','homeworks/hw2/tests/test_speed.py'],capture_output=True,text=True)
print(result.returncode,result.stdout,result.stderr)
print(subprocess.run(['python','-m','homeworks.hw2.numeric_eval','--model-dir','homeworks/hw2/model','--output','/content/quality/baseline_numeric.json'],capture_output=True,text=True).stdout)

1 ..F..                                                                    [100%]
=================================== FAILURES ===================================
___________ SpeedTests.test_inference_preserves_order_and_duplicates ___________

self = <test_speed.SpeedTests testMethod=test_inference_preserves_order_and_duplicates>

    def test_inference_preserves_order_and_duplicates(self):
        questions = ['long question ' * 10, 'short', 'short', 'medium question']
        with tempfile.TemporaryDirectory() as tmp:
            root = Path(tmp)
            pq.write_table(pa.table({'question': questions}), root / 'test.parquet')
            # Подмена только генерации: проверяем токенизацию, сортировку и запись.
            def generate(model, input_ids, **kwargs):
                return input_ids
            with patch('transformers.T5ForConditionalGeneration.generate', generate):
>               count, _ = generate_predictions(Path('homeworks/hw2/model'), root/'out.csv', root,
   

In [8]:
print(subprocess.run(['git','pull','--ff-only'],capture_output=True,text=True).stdout)
result = subprocess.run(['python','-m','pytest','-q','homeworks/hw2/tests/test_data.py','homeworks/hw2/tests/test_speed.py'],capture_output=True,text=True)
print(result.returncode,result.stdout,result.stderr)

Updating ec23fd4..b33fa54
Fast-forward
 homeworks/hw2/inference.py        | 5 ++---
 homeworks/hw2/tests/test_speed.py | 6 +++---
 2 files changed, 5 insertions(+), 6 deletions(-)

0 .....                                                                    [100%]
5 passed in 4.89s
 


In [9]:
from pathlib import Path
import subprocess, time
quality_root = Path('/content/quality')
quality_root.mkdir(exist_ok=True)
control_command = ['python','-u','-m','homeworks.hw2.train','--steps','6000','--batch-size','64','--accumulation','1','--precision','bf16','--group-window','32','--batches-per-shard','1000','--cache-dir','/content/hw2_cache','--output-dir','/content/quality/control','--save-every','500','--eval-every','500']
control_log = (quality_root/'control.log').open('w')
control_process = subprocess.Popen(control_command,stdout=control_log,stderr=subprocess.STDOUT)
print('CONTROL_PID',control_process.pid,'command',control_command)

CONTROL_PID 18407 command ['python', '-u', '-m', 'homeworks.hw2.train', '--steps', '6000', '--batch-size', '64', '--accumulation', '1', '--precision', 'bf16', '--group-window', '32', '--batches-per-shard', '1000', '--cache-dir', '/content/hw2_cache', '--output-dir', '/content/quality/control', '--save-every', '500', '--eval-every', '500']


In [15]:
print('CONTROL_STATUS', control_process.poll())
print((quality_root/'control.log').read_text()[-3000:])

CONTROL_STATUS 0
3000 val_token_loss=2.3095
best_step=3000 best_val_token_loss=2.3095
step=3050 train_loss=2.0665 elapsed_s=93.6
step=3100 train_loss=2.4547 elapsed_s=95.2
step=3150 train_loss=2.1654 elapsed_s=96.6
step=3200 train_loss=3.0146 elapsed_s=98.0
step=3250 train_loss=2.1121 elapsed_s=99.4
step=3300 train_loss=1.5625 elapsed_s=100.8
step=3350 train_loss=2.0871 elapsed_s=102.1
step=3400 train_loss=1.8924 elapsed_s=103.9
step=3450 train_loss=2.4253 elapsed_s=105.2
step=3500 train_loss=1.4615 elapsed_s=106.5
step=3500 val_token_loss=1.7333
best_step=3500 best_val_token_loss=1.7333
step=3550 train_loss=2.5375 elapsed_s=108.1
step=3600 train_loss=1.8210 elapsed_s=109.5
step=3650 train_loss=1.6010 elapsed_s=110.9
step=3700 train_loss=1.9316 elapsed_s=112.2
step=3750 train_loss=1.6311 elapsed_s=113.7
step=3800 train_loss=2.0383 elapsed_s=115.0
step=3850 train_loss=1.4170 elapsed_s=116.4
step=3900 train_loss=2.4131 elapsed_s=117.8
step=3950 train_loss=1.7379 elapsed_s=119.1
step=4000

In [11]:
import json, hashlib
summary['heldout_question_duplicates_in_remaining_shard0'] = sum(q in heldout_questions for q in table0[1024:])
summary['shard_samples'] = stats
(quality_root/'data_audit.json').write_text(json.dumps(summary,indent=2))
print('Saved data audit')
print('CONTROL_STATUS',control_process.poll())
print((quality_root/'control.log').read_text()[-1800:])

Saved data audit
CONTROL_STATUS None
erical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792072120.580056   18407 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792072121.785829   18407 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792072121.786094   18407 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
device=cuda precision=torch.bfloat16 group_window=32
parame

In [13]:
from concurrent.futures import ThreadPoolExecutor
shard_order = list(range(60)); random.Random(42).shuffle(shard_order)
# Avoid touching the six shards used by the running sequential control.
prefetch_indices = [i for i in range(60) if i not in shard_order[:7] and not (cache/f'part-{i:05d}.parquet').exists()]
def prefetch_one(index):
    return str(training_shard(index,cache))
prefetch_pool = ThreadPoolExecutor(max_workers=4)
prefetch_futures = [prefetch_pool.submit(prefetch_one,i) for i in prefetch_indices]
print('prefetch_started',len(prefetch_indices))
print('CONTROL_STATUS',control_process.poll())
print((quality_root/'control.log').read_text()[-1000:])

prefetch_started 49
CONTROL_STATUS None
0 train_loss=2.7051 elapsed_s=59.2
step=2000 val_token_loss=2.8233
best_step=2000 best_val_token_loss=2.8233
step=2050 train_loss=2.2110 elapsed_s=60.9
step=2100 train_loss=2.6049 elapsed_s=62.4
step=2150 train_loss=2.1466 elapsed_s=63.9
step=2200 train_loss=2.7285 elapsed_s=65.4
step=2250 train_loss=2.2988 elapsed_s=67.0
step=2300 train_loss=2.6591 elapsed_s=68.5
step=2350 train_loss=2.4923 elapsed_s=70.1
step=2400 train_loss=2.6838 elapsed_s=71.9
step=2450 train_loss=2.2393 elapsed_s=73.6
step=2500 train_loss=2.1635 elapsed_s=75.2
step=2500 val_token_loss=2.7443
best_step=2500 best_val_token_loss=2.7443
step=2550 train_loss=2.1939 elapsed_s=76.9
step=2600 train_loss=2.0299 elapsed_s=78.5
step=2650 train_loss=2.3150 elapsed_s=79.9
step=2700 train_loss=2.1711 elapsed_s=81.5
step=2750 train_loss=2.2661 elapsed_s=83.3
step=2800 train_loss=2.0045 elapsed_s=84.8
step=2850 train_loss=1.9434 elapsed_s=86.4
step=2900 train_loss=2.1312 elapsed_s=87.8
ste

In [14]:
print('CONTROL_STATUS',control_process.poll())
print((quality_root/'control.log').read_text()[-1100:])
print('PREFETCH',sum(f.done() for f in prefetch_futures),'/',len(prefetch_futures))
for f in prefetch_futures:
    if f.done() and f.exception(): print('PREFETCH_ERROR',repr(f.exception()))

CONTROL_STATUS None
p=2650 train_loss=2.3150 elapsed_s=79.9
step=2700 train_loss=2.1711 elapsed_s=81.5
step=2750 train_loss=2.2661 elapsed_s=83.3
step=2800 train_loss=2.0045 elapsed_s=84.8
step=2850 train_loss=1.9434 elapsed_s=86.4
step=2900 train_loss=2.1312 elapsed_s=87.8
step=2950 train_loss=1.7504 elapsed_s=89.3
step=3000 train_loss=2.2201 elapsed_s=92.0
step=3000 val_token_loss=2.3095
best_step=3000 best_val_token_loss=2.3095
step=3050 train_loss=2.0665 elapsed_s=93.6
step=3100 train_loss=2.4547 elapsed_s=95.2
step=3150 train_loss=2.1654 elapsed_s=96.6
step=3200 train_loss=3.0146 elapsed_s=98.0
step=3250 train_loss=2.1121 elapsed_s=99.4
step=3300 train_loss=1.5625 elapsed_s=100.8
step=3350 train_loss=2.0871 elapsed_s=102.1
step=3400 train_loss=1.8924 elapsed_s=103.9
step=3450 train_loss=2.4253 elapsed_s=105.2
step=3500 train_loss=1.4615 elapsed_s=106.5
step=3500 val_token_loss=1.7333
best_step=3500 best_val_token_loss=1.7333
step=3550 train_loss=2.5375 elapsed_s=108.1
step=3600 tr

In [16]:
assert control_process.poll() == 0
result = subprocess.run(['python','-m','homeworks.hw2.numeric_eval','--model-dir','/content/quality/control/best','--output','/content/quality/control_numeric.json'],capture_output=True,text=True)
print(result.returncode,result.stdout,result.stderr[-1000:]); result.check_returncode()
for f in prefetch_futures: f.result()
mixed_command = [x for x in control_command]
mixed_command[mixed_command.index('--batches-per-shard')] = '--mixed-rows-per-shard'
mixed_command[mixed_command.index('--mixed-rows-per-shard')+1] = '4096'
mixed_command[mixed_command.index('--output-dir')+1] = '/content/quality/mixed'
mixed_log = (quality_root/'mixed.log').open('w')
mixed_process = subprocess.Popen(mixed_command,stdout=mixed_log,stderr=subprocess.STDOUT)
print('MIXED_PID',mixed_process.pid,'command',mixed_command)

0 {
  "rows": 256,
  "usable_references": 245,
  "numeric_correct": 14,
  "numeric_agreement_on_usable": 0.05714285714285714,
  "prediction_failures": 46,
  "note": "Agreement with supplied train references; not public pass@1. Incorrect references and restricted execution may affect this diagnostic.",
  "model_dir": "/content/quality/control/best",
  "offset": 256,
  "beams": 1
}
 erent numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792072324.946159   19518 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792072326.163458   19518 port.cc:153] oneDNN custom operations are on. You may see slightly dif

In [19]:
print('MIXED_STATUS', mixed_process.poll())
print((quality_root/'mixed.log').read_text()[-1600:])

MIXED_STATUS None
00 best_val_token_loss=1.9518
step=2550 train_loss=2.3548 elapsed_s=77.8
step=2600 train_loss=1.5646 elapsed_s=79.5
step=2650 train_loss=2.4013 elapsed_s=80.9
step=2700 train_loss=1.7380 elapsed_s=82.3
step=2750 train_loss=2.3687 elapsed_s=83.7
step=2800 train_loss=1.8225 elapsed_s=85.1
step=2850 train_loss=1.1534 elapsed_s=86.6
step=2900 train_loss=2.1583 elapsed_s=87.9
step=2950 train_loss=1.3242 elapsed_s=89.7
step=3000 train_loss=2.0463 elapsed_s=91.1
step=3000 val_token_loss=1.7993
best_step=3000 best_val_token_loss=1.7993
step=3050 train_loss=1.4326 elapsed_s=92.7
step=3100 train_loss=2.1972 elapsed_s=94.0
step=3150 train_loss=1.6572 elapsed_s=95.3
step=3200 train_loss=2.6467 elapsed_s=96.6
step=3250 train_loss=1.7358 elapsed_s=98.2
step=3300 train_loss=1.3976 elapsed_s=99.6
step=3350 train_loss=1.9803 elapsed_s=100.9
step=3400 train_loss=1.3894 elapsed_s=102.3
step=3450 train_loss=2.1252 elapsed_s=103.6
step=3500 train_loss=1.5699 elapsed_s=105.0
step=3500 val_

In [20]:
print('MIXED_STATUS',mixed_process.poll())
print((quality_root/'mixed.log').read_text()[-1200:])
print('cached_shards',len(list(cache.glob('part-*.parquet'))))

MIXED_STATUS 0
train_loss=1.6696 elapsed_s=152.2
step=5000 train_loss=1.1577 elapsed_s=153.5
step=5000 val_token_loss=1.5170
best_step=5000 best_val_token_loss=1.5170
step=5050 train_loss=1.9179 elapsed_s=155.1
step=5100 train_loss=1.3804 elapsed_s=156.6
step=5150 train_loss=2.0101 elapsed_s=157.9
step=5200 train_loss=1.5655 elapsed_s=159.3
step=5250 train_loss=0.9593 elapsed_s=160.9
step=5300 train_loss=1.6576 elapsed_s=162.3
step=5350 train_loss=1.2068 elapsed_s=163.8
step=5400 train_loss=1.7598 elapsed_s=165.1
step=5450 train_loss=1.4656 elapsed_s=166.6
step=5500 train_loss=1.8983 elapsed_s=167.9
step=5500 val_token_loss=1.4944
best_step=5500 best_val_token_loss=1.4944
step=5550 train_loss=1.5705 elapsed_s=169.5
step=5600 train_loss=2.3040 elapsed_s=170.8
step=5650 train_loss=1.4695 elapsed_s=172.6
step=5700 train_loss=1.2306 elapsed_s=173.9
step=5750 train_loss=1.6478 elapsed_s=175.2
step=5800 train_loss=1.3405 elapsed_s=176.6
step=5850 train_loss=1.8297 elapsed_s=178.0
step=5900 t

In [21]:
assert mixed_process.poll() == 0
result = subprocess.run(['python','-m','homeworks.hw2.numeric_eval','--model-dir','/content/quality/mixed/best','--output','/content/quality/mixed_numeric.json'],capture_output=True,text=True)
print(result.returncode,result.stdout,result.stderr[-300:]); result.check_returncode()
lr_command = mixed_command.copy()
lr_command[lr_command.index('--output-dir')+1] = '/content/quality/mixed_lr001'
lr_command += ['--learning-rate','0.001']
lr_log = (quality_root/'mixed_lr001.log').open('w')
lr_process = subprocess.Popen(lr_command,stdout=lr_log,stderr=subprocess.STDOUT)
print('LR_PID',lr_process.pid,'command',lr_command)

0 {
  "rows": 256,
  "usable_references": 245,
  "numeric_correct": 16,
  "numeric_agreement_on_usable": 0.0653061224489796,
  "prediction_failures": 42,
  "note": "Agreement with supplied train references; not public pass@1. Incorrect references and restricted execution may affect this diagnostic.",
  "model_dir": "/content/quality/mixed/best",
  "offset": 256,
  "beams": 1
}
 different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792072540.129302   21409 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

LR_PID 22189 command ['python', '-u', '-m', 'homeworks.hw2.train', '--steps', '6000', '--batch-size', '64', '--accumulation', '1', '--precision', 'bf16', '--group-window', '32', '--mixed-rows-per-shard', '4096', '--cache-dir', '/content/hw2_cache', '--output-dir', '/content/quality/mixed_lr001', '--save-every', '500'

In [24]:
print('LR_STATUS',lr_process.poll())
print((quality_root/'mixed_lr001.log').read_text()[-1200:])
print('selection_diagnostic_question_overlap',len({r['question'] for r in rows[:256]} & {r['question'] for r in rows[256:512]}))

LR_STATUS None
=4250 train_loss=1.5417 elapsed_s=131.4
step=4300 train_loss=1.2117 elapsed_s=133.1
step=4350 train_loss=1.5738 elapsed_s=134.5
step=4400 train_loss=1.1947 elapsed_s=135.9
step=4450 train_loss=0.7197 elapsed_s=137.2
step=4500 train_loss=1.3151 elapsed_s=138.5
step=4500 val_token_loss=1.1984
best_step=4500 best_val_token_loss=1.1984
step=4550 train_loss=1.1095 elapsed_s=140.1
step=4600 train_loss=1.5237 elapsed_s=141.8
step=4650 train_loss=1.0560 elapsed_s=143.2
step=4700 train_loss=1.4828 elapsed_s=144.6
step=4750 train_loss=1.4009 elapsed_s=146.1
step=4800 train_loss=1.7355 elapsed_s=147.4
step=4850 train_loss=1.4049 elapsed_s=148.8
step=4900 train_loss=1.1034 elapsed_s=150.2
step=4950 train_loss=1.3262 elapsed_s=151.8
step=5000 train_loss=0.9002 elapsed_s=153.2
step=5000 val_token_loss=1.1693
best_step=5000 best_val_token_loss=1.1693
step=5050 train_loss=1.4168 elapsed_s=154.8
step=5100 train_loss=1.1195 elapsed_s=156.1
step=5150 train_loss=1.5418 elapsed_s=157.5
step=

In [23]:
import csv
from homeworks.hw2.numeric_eval import execute_numeric
with open('homeworks/hw2/predictions.csv',newline='') as f:
    historical_codes = [r['code'] for r in csv.DictReader(f)]
# Check alignment evidence using only bot-reported question text and OUR output.
# Do not use the bot's gold values or modify any answer.
alignment_samples = [
    ('Cindy paid $54 for a sweater after a 10% sales tax was added.',1.4),
    ('Student: The area of a triangle is 54 square inches.',324.0),
    ('A restaurant sells 200 burgers on a regular day.',8.0),
]
alignment = []
for prefix, reported_own_value in alignment_samples:
    indices = [i for i,r in enumerate(test) if r['question'].startswith(prefix)]
    values = [execute_numeric(historical_codes[i]) for i in indices]
    alignment.append({'question_prefix':prefix,'matching_parquet_indices':indices,'historical_code_values':values,'bot_reported_our_value':reported_own_value})
(quality_root/'sampled_alignment.json').write_text(json.dumps(alignment,indent=2))
print(json.dumps(alignment,indent=2))

[
  {
    "question_prefix": "Cindy paid $54 for a sweater after a 10% sales tax was added.",
    "matching_parquet_indices": [
      0
    ],
    "historical_code_values": [
      {
        "status": "ok",
        "value": 1.4000000000000001
      }
    ],
    "bot_reported_our_value": 1.4
  },
  {
    "question_prefix": "Student: The area of a triangle is 54 square inches.",
    "matching_parquet_indices": [
      1
    ],
    "historical_code_values": [
      {
        "status": "ok",
        "value": 324.0
      }
    ],
    "bot_reported_our_value": 324.0
  },
  {
    "question_prefix": "A restaurant sells 200 burgers on a regular day.",
    "matching_parquet_indices": [
      2
    ],
    "historical_code_values": [
      {
        "status": "ok",
        "value": 8.0
      }
    ],
    "bot_reported_our_value": 8.0
  }
]


In [25]:
lr_process.wait(timeout=30)
result = subprocess.run(['python','-m','homeworks.hw2.numeric_eval','--model-dir','/content/quality/mixed_lr001/best','--output','/content/quality/mixed_lr001_numeric.json'],capture_output=True,text=True)
print(result.returncode,result.stdout,result.stderr[-300:]); result.check_returncode()
import re, hashlib
quality_results = {'date':'2026-10-15','training_source_commit':'b33fa54','gpu':torch.cuda.get_device_name(0),'runs':{}}
for name in ['control','mixed','mixed_lr001']:
    root = quality_root/name
    log = (quality_root/f'{name}.log').read_text()
    quality_results['runs'][name] = {'best_validation':json.loads((root/'best/validation.json').read_text()),'numeric':json.loads((quality_root/f'{name}_numeric.json').read_text()),'run_config':json.loads((root/'run_config.json').read_text()),'model_sha256':hashlib.sha256((root/'best/model.safetensors').read_bytes()).hexdigest(),'validation_curve':[{'step':int(a),'token_loss':float(b)} for a,b in re.findall(r'step=(\d+) val_token_loss=([0-9.]+)',log)]}
(quality_root/'quality_results.json').write_text(json.dumps(quality_results,indent=2))
print(json.dumps(quality_results,indent=2))

0 {
  "rows": 256,
  "usable_references": 245,
  "numeric_correct": 27,
  "numeric_agreement_on_usable": 0.11020408163265306,
  "prediction_failures": 22,
  "note": "Agreement with supplied train references; not public pass@1. Incorrect references and restricted execution may affect this diagnostic.",
  "model_dir": "/content/quality/mixed_lr001/best",
  "offset": 256,
  "beams": 1
}
 different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792072746.018463   23260 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

{
  "date": "2026-10-15",
  "training_source_commit": "b33fa54",
  "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "runs": {
    "control": {
      "best_validation": {
        "step": 4000,
        "token_loss": 1.6011834497744253
      },
      "numeric": {
        "rows": 256,
        "usable_refer

In [26]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
archive_path = Path('/content/hw2_quality_pilots_20261015.zip')
with ZipFile(archive_path,'w',compression=ZIP_DEFLATED,compresslevel=1) as archive:
    for path in quality_root.rglob('*'):
        if not path.is_file(): continue
        if '/latest/' in str(path) and '/mixed_lr001/' not in str(path): continue
        archive.write(path,path.relative_to(quality_root))
print('ARCHIVE',archive_path.stat().st_size,'SHA256',hashlib.sha256(archive_path.read_bytes()).hexdigest())
files.download(str(archive_path))

ARCHIVE 221498788 SHA256 65cfd26f0f0b537972d0d6e95af16e6d8621fafccb85a22c8539d9b181c1d080


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
assert lr_process.poll() == 0
long_command = lr_command.copy()
long_command[long_command.index('--steps')+1] = '20000'
long_command[long_command.index('--output-dir')+1] = '/content/quality/mixed_lr001_20k'
long_log = (quality_root/'mixed_lr001_20k.log').open('w')
long_process = subprocess.Popen(long_command,stdout=long_log,stderr=subprocess.STDOUT)
print('LONG_PID',long_process.pid,'command',long_command)

LONG_PID 24767 command ['python', '-u', '-m', 'homeworks.hw2.train', '--steps', '20000', '--batch-size', '64', '--accumulation', '1', '--precision', 'bf16', '--group-window', '32', '--mixed-rows-per-shard', '4096', '--cache-dir', '/content/hw2_cache', '--output-dir', '/content/quality/mixed_lr001_20k', '--save-every', '500', '--eval-every', '500', '--learning-rate', '0.001']


In [ ]:
print('LONG_STATUS',long_process.poll())
print((quality_root/'mixed_lr001_20k.log').read_text()[-1000:])